# Audit saved membership only
No model training. Derived from the original saved-splits audit cells. Attach the immutable original CSV and images; never rerun fold assignment.
CPU is sufficient. Verify paths; preserve an existing audited output before recreating it.


In [ ]:
# 2. Paths and settings
import os,json,random,hashlib,sqlite3,math,importlib.metadata,gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch,torchvision

SPLITS_PATH=Path('/kaggle/input/datasets/feras2004/spliteoriginal/splitsOriginal.csv')
BENIGN=Path('/kaggle/input/datasets/feras2004/qr-all-benign/QR_All_benign/qrs')
MALICIOUS=Path('/kaggle/input/datasets/feras2004/qr-all-benign/QR_All_Malicious/qrs')
OUT=Path('/kaggle/working/qr_visual_fresh'); OUT.mkdir(parents=True,exist_ok=True)
CLEAN_DUPLICATES=True
PREPARED_DIR=None  # after first audit: '/kaggle/working/qr_visual_fresh'
# Or attach saved output as a Kaggle Dataset and use its audit/parquet directory.
# Only reuse prepared data when the QR image Dataset version is unchanged.
FAST_LOAD=False
HASH_WORKERS=min(4,os.cpu_count() or 1)
# Optional: resume hashing from a saved output attached as a Kaggle Dataset.
RESTORE_HASH_CACHE=None  # absolute path to saved hashes.sqlite

assert SPLITS_PATH.is_file(),f'Attach splits Dataset: {SPLITS_PATH}'
assert BENIGN.is_dir() and MALICIOUS.is_dir(),'Attach the QR image Dataset.'
versions={k:importlib.metadata.version(k) for k in
    ['torch','torchvision','scikit-learn','numpy','pandas']}
(OUT/'environment.json').write_text(json.dumps(versions,indent=2))
print(versions)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

SOURCE_SPLITS_SHA256=hashlib.sha256(SPLITS_PATH.read_bytes()).hexdigest()
if PREPARED_DIR is not None:
    prepared=Path(PREPARED_DIR)
    saved_audit=json.loads((prepared/'audit.json').read_text())
    assert saved_audit.get('source_splits_sha256')==SOURCE_SPLITS_SHA256,         'Prepared data belongs to a different/older source CSV; run full audit once.'
    assert saved_audit.get('image_roots')==[str(BENIGN),str(MALICIOUS)]
    assert saved_audit.get('clean_duplicates')==CLEAN_DUPLICATES
    assert saved_audit.get('train_ready') is True
    prepared_file=prepared/'splits_clean.parquet'
    assert hashlib.sha256(prepared_file.read_bytes()).hexdigest()==saved_audit['parquet_sha256']
    df=pd.read_parquet(prepared_file)
    assert set(df.split)=={'train','val','test'} and df.path.is_unique
    assert df.groupby('split').label.nunique().eq(2).all()
    assert df.groupby('domain',dropna=False).split.nunique().max()==1
    assert df.groupby('sha256').split.nunique().max()==1
    DATA_SIGNATURE=saved_audit['data_signature']; TRAIN_READY=True; FAST_LOAD=True
    print('Prepared manifest loaded: skipped per-image checking and hashing.')
    print(pd.crosstab(df.split,df.label))

if not FAST_LOAD:
    # 3. Load SAVED membership. Never derive a new hostname or fold.
    df_raw=pd.read_csv(SPLITS_PATH,keep_default_na=False,
        dtype={'path':str,'url':str,'domain':str,'split':str})
    required={'path','url','ok','label','domain','fold','split'}
    assert required.issubset(df_raw.columns)
    assert len(df_raw)==1_005_738, 'Original source row count differs; stop and inspect.'
    assert not df_raw.path.duplicated().any()
    for col,allowed in [('label',[0,1]),('ok',[0,1]),('fold',list(range(20)))]:
        values=pd.to_numeric(df_raw[col],errors='coerce')
        invalid=~values.isin(allowed)
        if invalid.any():
            print(df_raw.loc[invalid,['path',col]].head(5))
            raise ValueError(f'Invalid {col}: {int(invalid.sum())} rows; stop before training.')
        df_raw[col]=values.astype('int64')
    assert set(df_raw.split)=={'train','val','test'}
    assert df_raw.fold.between(0,19).all()
    expected=np.where(df_raw.fold.lt(3),'test',np.where(df_raw.fold.lt(6),'val','train'))
    assert np.array_equal(expected,df_raw.split.to_numpy()),'Fold/split inconsistency.'

    # Resolve image paths by class directory and relative filename, not filename prefix.
    # Some files named benign_* belong to the malicious directory; labels stay as saved.
    def rebind_path(row):
        root=BENIGN if int(row.label)==0 else MALICIOUS
        marker=('QR_All_benign/qrs/' if int(row.label)==0 else 'QR_All_Malicious/qrs/')
        assert marker in row.path,'Saved directory and label disagree: '+row.path
        relative=row.path.split(marker,1)[1]
        assert not Path(relative).is_absolute() and '..' not in Path(relative).parts
        return str(root/relative)

    df_raw['source_path']=df_raw.path
    mapped=[rebind_path(r) for r in df_raw.itertuples(index=False)]
    # Preserve failed-decode grouping identity when rebinding image locations.
    failed=df_raw.ok.eq(0)
    assert (df_raw.loc[failed,'domain']==df_raw.loc[failed,'source_path']).all(),     'Unexpected failed-decode group convention; inspect before proceeding.'
    df_raw['path']=mapped
    assert len(df_raw)==1_005_738, 'Original source row count differs; stop and inspect.'
    assert not df_raw.path.duplicated().any()
    print(pd.crosstab(df_raw.split,df_raw.label).reindex(['train','val','test']))
    print('Empty saved domain:',int(df_raw.domain.eq('').sum()))
    print('Decode failures:',int(df_raw.ok.eq(0).sum()))

else:
    print("Using previously audited saved manifest.")

if not FAST_LOAD:
    # 4. Resumable SHA256 + image readability audit.
    from PIL import Image
    import shutil
    cache=OUT/'hashes.sqlite'
    if RESTORE_HASH_CACHE is not None and not cache.exists():
        shutil.copy2(RESTORE_HASH_CACHE,cache)
    con=sqlite3.connect(cache)
    con.execute('PRAGMA synchronous=FULL')
    assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok', 'Hash cache is invalid: do not use it.'
    con.execute('CREATE TABLE IF NOT EXISTS hashes '
                '(path TEXT PRIMARY KEY,size INTEGER,mtime INTEGER,sha256 TEXT,error TEXT)')
    con.commit()
    known={p:(s,m,h,e) for p,s,m,h,e in con.execute('SELECT * FROM hashes')}
    rows=[]; pending=[]; missing_count=0; missing_examples=[]
    for p in tqdm(df_raw.path,desc='Check files and cache'):
        path=Path(p)
        if not path.is_file():
            missing_count+=1
            if len(missing_examples)<10: missing_examples.append(p)
            continue
        stat=path.stat(); previous=known.get(p)
        if previous and previous[:2]==(stat.st_size,stat.st_mtime_ns) and not previous[3]:
            rows.append((p,*previous))
        else: pending.append(p)
    assert missing_count==0,f'{missing_count} missing images; examples: {missing_examples}'

    def hash_image(p):
        path=Path(p); stat=path.stat(); digest=hashlib.sha256()
        error=''
        try:
            with path.open('rb') as f:
                for chunk in iter(lambda:f.read(1024*1024),b''): digest.update(chunk)
            # Decode image data to catch corrupt files before training.
            with Image.open(path) as im: im.load()
        except Exception as exc: error=type(exc).__name__+': '+str(exc)[:200]
        return p,stat.st_size,stat.st_mtime_ns,digest.hexdigest(),error

    with ThreadPoolExecutor(max_workers=HASH_WORKERS) as pool:
        for start in tqdm(range(0,len(pending),2000),desc='Hash batches'):
            batch=list(pool.map(hash_image,pending[start:start+2000]))
            con.executemany('INSERT OR REPLACE INTO hashes VALUES (?,?,?,?,?)',batch)
            con.commit(); rows.extend(batch)
    con.close()
    hashes=pd.DataFrame(rows,columns=['path','size','mtime','sha256','image_error'])
    assert len(hashes)==len(df_raw) and hashes.path.is_unique
    bad=hashes[hashes.image_error.ne('')]
    bad.to_csv(OUT/'image_errors.csv',index=False)
    assert bad.empty,f'Review {len(bad)} invalid images in image_errors.csv.'
    df_raw=df_raw.merge(hashes[['path','sha256']],on='path',how='left',validate='one_to_one')
    assert df_raw.sha256.notna().all()

else:
    print("Using previously audited saved manifest.")

if not FAST_LOAD:
    # 5. Audit, clean, and preserve saved memberships.
    stats=df_raw.groupby('sha256').agg(n=('path','size'),labels=('label','nunique'))
    conflicts=set(stats.index[stats.labels.gt(1)])
    df_raw[df_raw.sha256.isin(conflicts)].to_csv(OUT/'label_conflicts.csv',index=False)
    df_raw[df_raw.sha256.isin(stats.index[stats.n.gt(1)])].to_csv(OUT/'byte_duplicates.csv',index=False)
    if CLEAN_DUPLICATES:
        df=df_raw[~df_raw.sha256.isin(conflicts)].copy()
        df['_priority']=df.split.map({'train':0,'val':1,'test':2})
        df=df.sort_values(['_priority','path']).drop_duplicates('sha256').drop(columns='_priority')
    else: df=df_raw.copy()
    df=df.sort_values('path').reset_index(drop=True)
    removed=df_raw[~df_raw.path.isin(df.path)]
    removed.to_csv(OUT/'removed_rows.csv',index=False)
    assert df.groupby('split').label.nunique().eq(2).all()
    # Empty strings remain one saved group; they are not dropped by groupby.
    host_cross=int(df.groupby('domain',dropna=False).split.nunique().gt(1).sum())
    hash_cross=int(df.groupby('sha256').split.nunique().gt(1).sum())
    url_cross=int(df[df.ok.eq(1)].groupby('url').split.nunique().gt(1).sum())
    retained_conflicts=int(df.groupby('sha256').label.nunique().gt(1).sum())
    TRAIN_READY=all(v==0 for v in [host_cross,hash_cross,url_cross,retained_conflicts])
    # Deterministic identity binds every trained checkpoint to exact paths/labels/splits/URLs.
    signature=hashlib.sha256()
    for r in df[['path','label','split','url','ok','sha256']].itertuples(index=False,name=None):
        signature.update((json.dumps(r,ensure_ascii=False,separators=(',',':'))+'\n').encode())
    DATA_SIGNATURE=signature.hexdigest()
    audit=dict(original_samples=len(df_raw),retained_samples=len(df),removed_samples=len(removed),
               clean_duplicates=CLEAN_DUPLICATES,conflicting_hash_groups_original=len(conflicts),
               host_groups_across_splits=host_cross,byte_hashes_across_splits=hash_cross,
               decoded_urls_across_splits=url_cross,conflicts_retained=retained_conflicts,
               train_ready=TRAIN_READY,data_signature=DATA_SIGNATURE,
               membership='saved CSV; no reassignment')
    summary=pd.crosstab(df.split,df.label).reindex(['train','val','test'])
    summary['total']=summary.sum(axis=1); summary['fraction']=summary.total/len(df)
    print(summary); print(audit)
    summary.to_csv(OUT/'split_summary.csv')
    parquet_tmp=OUT/'splits_clean.writing.parquet'
    df.to_parquet(parquet_tmp,index=False)
    import pyarrow.parquet as pq
    assert pq.ParquetFile(parquet_tmp).metadata.num_rows==len(df)
    os.replace(parquet_tmp,OUT/'splits_clean.parquet')
    audit['source_splits_sha256']=SOURCE_SPLITS_SHA256
    audit['image_roots']=[str(BENIGN),str(MALICIOUS)]
    audit['parquet_sha256']=hashlib.sha256((OUT/'splits_clean.parquet').read_bytes()).hexdigest()
    (OUT/'audit.writing.json').write_text(json.dumps(audit,indent=2))
    os.replace(OUT/'audit.writing.json',OUT/'audit.json')
    # Free audit-only memory before allocating model weights.
    del known,rows,pending,hashes,stats,removed,df_raw,mapped
    if 'batch' in globals(): del batch
    gc.collect()
    assert TRAIN_READY,'Resolve audit conflicts/leaks before training; see audit.json.'

else:
    print("Using previously audited saved manifest.")
